In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support, f1_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)
tf.keras.mixed_precision.set_global_policy('float32')

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

TensorFlow version: 2.19.0
GPU available: True


In [49]:
df = pd.read_csv("authentic_iot_sensor_data.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print("Dataset shape:", df.shape)
print("\nClass distribution:\n", df['anomaly_type'].value_counts())
anomaly_ratio = (df['anomaly_type'] == 'anomaly').mean()
print(f"\nAnomaly ratio: {anomaly_ratio:.2%}")
df.head()

Dataset shape: (3000, 8)

Class distribution:
 anomaly_type
normal     2826
anomaly     174
Name: count, dtype: int64

Anomaly ratio: 5.80%


,timestamp,temperature,humidity,lux,accel_x,accel_y,accel_z,anomaly_type
0,2026-04-20 14:48:25,29.73799,41.08180,320.30557,-0.00499,-0.06936,9.79785,normal
1,2026-04-20 14:48:29,29.86857,41.19705,321.24008,0.03543,0.01087,9.83533,normal
2,2026-04-20 14:48:33,29.78596,41.61364,321.74757,0.04259,0.03038,9.85397,normal
3,2026-04-20 14:48:37,29.65000,41.59543,322.02826,-0.03463,-0.04396,9.79249,normal
4,2026-04-20 14:48:41,29.78323,41.61522,323.51225,-0.01078,-0.06166,9.78505,normal


In [50]:
features = ['temperature', 'humidity', 'lux', 'accel_x', 'accel_y', 'accel_z']
X = df[features].values
y = (df['anomaly_type'] == 'anomaly').astype(int).values

train_mask = (y == 0)
X_train_raw = X[train_mask]
X_test_raw = X
y_test = y

print(f"Training samples (normal only): {X_train_raw.shape[0]}")
print(f"Test samples (all): {X_test_raw.shape[0]}")
print(f"Test anomalies: {y_test.sum()}")

Training samples (normal only): 2826
Test samples (all): 3000
Test anomalies: 174


In [51]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(X_train_raw)

X_train = scaler.transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

eps = 1e-8
X_train = np.clip(X_train, eps, 1 - eps)
X_test = np.clip(X_test, eps, 1 - eps)

In [52]:
WINDOW_SIZE = 20          # increased from 10

def create_windows(data, window_size):
    n = len(data)
    windows = []
    first_row = data[0]
    for i in range(n):
        if i < window_size:
            pad_count = window_size - i - 1
            pad_rows = np.tile(first_row, (pad_count, 1))
            window = np.vstack([pad_rows, data[:i+1]])
        else:
            window = data[i-window_size+1 : i+1]
        windows.append(window)
    return np.array(windows, dtype=np.float32)

train_windows = create_windows(X_train, WINDOW_SIZE)
test_windows = create_windows(X_test, WINDOW_SIZE)

print("Train windows shape:", train_windows.shape)
print("Test windows shape:", test_windows.shape)

Train windows shape: (2826, 20, 6)
Test windows shape: (3000, 20, 6)


In [53]:
CONTEXT_LEN = 40          # twice window size

def create_contexts(data, context_len):
    n = len(data)
    contexts = []
    first_row = data[0]
    for i in range(n):
        if i < context_len:
            pad_count = context_len - i - 1
            pad_rows = np.tile(first_row, (pad_count, 1))
            ctx = np.vstack([pad_rows, data[:i+1]])
        else:
            ctx = data[i-context_len+1 : i+1]
        contexts.append(ctx)
    return np.array(contexts, dtype=np.float32)

train_contexts = create_contexts(X_train, CONTEXT_LEN)
test_contexts = create_contexts(X_test, CONTEXT_LEN)

print("Train contexts shape:", train_contexts.shape)
print("Test contexts shape:", test_contexts.shape)

Train contexts shape: (2826, 40, 6)
Test contexts shape: (3000, 40, 6)


In [54]:
assert len(test_windows) == len(y_test)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
for val_idx, test_idx in sss.split(test_windows, y_test):
    val_windows = test_windows[val_idx]
    val_contexts = test_contexts[val_idx]
    val_labels = y_test[val_idx]

    final_test_windows = test_windows[test_idx]
    final_test_contexts = test_contexts[test_idx]
    final_test_labels = y_test[test_idx]

print("Validation set size:", len(val_labels))
print("Anomalies in validation:", val_labels.sum())
print("Final test set size:", len(final_test_labels))
print("Anomalies in final test:", final_test_labels.sum())

Validation set size: 1500
Anomalies in validation: 87
Final test set size: 1500
Anomalies in final test: 87


In [55]:
def build_tranad_model(window_size=20, context_len=40, num_features=6,
                       num_heads=6, key_dim=10, ff_dim=64, dropout=0.1):
    W_input = layers.Input(shape=(window_size, num_features), name='W')
    C_input = layers.Input(shape=(context_len, num_features), name='C')
    F_input = layers.Input(shape=(window_size, num_features), name='F')

    d_model = 32          # reduced for bottleneck

    # ---- Context Encoder ----
    attn1 = layers.MultiHeadAttention(key_dim=key_dim, num_heads=num_heads, dropout=dropout)
    x1 = attn1(C_input, C_input)
    x1 = layers.Dropout(dropout)(x1)
    x1 = layers.Add()([x1, C_input])
    x1 = layers.LayerNormalization(epsilon=1e-6)(x1)

    ff1 = layers.Dense(ff_dim, activation='relu')(x1)
    ff1 = layers.Dense(num_features)(ff1)
    ff1 = layers.Add()([ff1, x1])
    ff1 = layers.LayerNormalization(epsilon=1e-6)(ff1)

    ctx_proj = layers.Dense(d_model, name='ctx_proj')(ff1)
    ctx_proj = layers.LayerNormalization(epsilon=1e-6)(ctx_proj)
    enc_context = ctx_proj

    # ---- Window Encoder (masked) ----
    wf_concat = layers.Concatenate(axis=-1, name='wf_concat')([W_input, F_input])

    attn2 = layers.MultiHeadAttention(key_dim=key_dim, num_heads=num_heads, dropout=dropout)
    lookahead_mask = 1 - tf.linalg.band_part(tf.ones((window_size, window_size)), -1, 0)
    x2 = attn2(wf_concat, wf_concat, attention_mask=lookahead_mask)
    x2 = layers.Dropout(dropout)(x2)
    x2 = layers.Add()([x2, wf_concat])
    x2 = layers.LayerNormalization(epsilon=1e-6)(x2)

    win_proj = layers.Dense(d_model, name='win_proj')(x2)
    win_proj = layers.LayerNormalization(epsilon=1e-6)(win_proj)

    # Cross‑attention
    cross_attn = layers.MultiHeadAttention(key_dim=key_dim, num_heads=num_heads, dropout=dropout)
    x_cross = cross_attn(query=win_proj, value=enc_context, key=enc_context)
    x_cross = layers.Dropout(dropout)(x_cross)
    x_cross = layers.Add()([x_cross, win_proj])
    x_cross = layers.LayerNormalization(epsilon=1e-6)(x_cross)

    ff2 = layers.Dense(ff_dim, activation='relu')(x_cross)
    ff2 = layers.Dense(d_model)(ff2)
    ff2 = layers.Add()([ff2, x_cross])
    ff2 = layers.LayerNormalization(epsilon=1e-6)(ff2)
    enc_window = ff2

    # ---- Decoders (only first is used for scoring) ----
    def make_decoder(name):
        h = layers.Dense(d_model, activation='relu', name=f'{name}_dense1')(enc_window)
        out = layers.Dense(num_features, activation='sigmoid', name=f'{name}_output')(h)
        return out

    O1 = make_decoder('decoder1')
    O2 = make_decoder('decoder2')

    return Model(inputs=[W_input, C_input, F_input], outputs=[O1, O2], name='TranAD')

NUM_FEATURES = len(features)
NUM_HEADS = 6
KEY_DIM = 10
FF_DIM = 64
DROPOUT = 0.1

model = build_tranad_model(window_size=WINDOW_SIZE, context_len=CONTEXT_LEN,
                           num_features=NUM_FEATURES, num_heads=NUM_HEADS,
                           key_dim=KEY_DIM, ff_dim=FF_DIM, dropout=DROPOUT)
model.summary()

Model: "TranAD"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ C (InputLayer)      │ (None, 40, 6)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 40, 6)     │      1,626 │ C[0][0], C[0][0]  │
│ (MultiHeadAttentio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 40, 6)     │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_15 (Add)        │ (None, 40, 6)     │          0 │ dropout_19[0][0], │
│                     │                   │            │ C[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ W (InputLayer)      │ (None, 20, 6)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ F (InputLayer)      │ (None, 20, 6)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 40, 6)     │         12 │ add_15[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ wf_concat           │ (None, 20, 12)    │          0 │ W[0][0], F[0][0]  │
│ (Concatenate)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 40, 64)    │        448 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 20, 12)    │      3,072 │ wf_concat[0][0],  │
│ (MultiHeadAttentio… │                   │            │ wf_concat[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 40, 6)     │        390 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_21          │ (None, 20, 12)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_16 (Add)        │ (None, 40, 6)     │          0 │ dense_13[0][0],   │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_17 (Add)        │ (None, 20, 12)    │          0 │ dropout_21[0][0], │
│                     │                   │            │ wf_concat[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 40, 6)     │         12 │ add_16[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 12)    │         24 │ add_17[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ctx_proj (Dense)    │ (None, 40, 32)    │        224 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ win_proj (Dense)    │ (None, 20, 32)    │        416 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 40, 32)    │         64 │ ctx_proj[0][0]    │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 21,072 (82.31 KB)

 Trainable params: 21,072 (82.31 KB)

 Non-trainable params: 0 (0.00 B)

In [56]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    (train_windows, train_contexts),
    train_windows
)).batch(64).prefetch(tf.data.AUTOTUNE)

In [57]:
def train_step(batch_W, batch_C):
    focus_zeros = tf.zeros_like(batch_W)
    with tf.GradientTape() as tape:
        O1, _ = model([batch_W, batch_C, focus_zeros], training=True)
        loss = tf.reduce_mean(tf.square(O1 - batch_W))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

In [58]:
def predict_anomaly_scores(model, test_windows, test_contexts, batch_size=64):
    n = test_windows.shape[0]
    scores = np.zeros(n)
    for i in range(0, n, batch_size):
        batch_W = test_windows[i:i+batch_size]
        batch_C = test_contexts[i:i+batch_size]
        focus_zeros = np.zeros_like(batch_W)
        O1, _ = model.predict_on_batch([batch_W, batch_C, focus_zeros])
        scores[i:i+batch_size] = np.mean((O1 - batch_W)**2, axis=(1,2))
    return scores

In [59]:
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.005, decay_steps=1000, decay_rate=0.9, staircase=True
)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-5)

def compute_val_f1(model, val_w, val_c, val_y, percentile=95):
    scores = predict_anomaly_scores(model, val_w, val_c)
    th = np.percentile(scores, percentile)
    pred = (scores > th).astype(int)
    return f1_score(val_y, pred)

EPOCHS = 100
patience = 10
best_val_f1 = 0
epochs_no_improve = 0

train_losses = []
val_f1_scores = []

for epoch in range(1, EPOCHS+1):
    epoch_loss = []
    for (batch_W, batch_C), _ in train_dataset:
        loss = train_step(batch_W, batch_C)
        epoch_loss.append(loss.numpy())
    avg_loss = np.mean(epoch_loss)
    train_losses.append(avg_loss)

    val_f1 = compute_val_f1(model, val_windows, val_contexts, val_labels, percentile=95)
    val_f1_scores.append(val_f1)

    print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.6f} | Val F1 (95%): {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping. Best val F1: {best_val_f1:.4f}")
            break

Epoch   1/100 | Loss: 0.031364 | Val F1 (95%): 0.1605
Epoch   2/100 | Loss: 0.040955 | Val F1 (95%): 0.1975
Epoch   3/100 | Loss: 0.032823 | Val F1 (95%): 0.2346
Epoch   4/100 | Loss: 0.035251 | Val F1 (95%): 0.1852
Epoch   5/100 | Loss: 0.032699 | Val F1 (95%): 0.1975
Epoch   6/100 | Loss: 0.032009 | Val F1 (95%): 0.2099
Epoch   7/100 | Loss: 0.031743 | Val F1 (95%): 0.2099
Epoch   8/100 | Loss: 0.031617 | Val F1 (95%): 0.2099
Epoch   9/100 | Loss: 0.031557 | Val F1 (95%): 0.2099
Epoch  10/100 | Loss: 0.031529 | Val F1 (95%): 0.2222
Epoch  11/100 | Loss: 0.031516 | Val F1 (95%): 0.2222
Epoch  12/100 | Loss: 0.031512 | Val F1 (95%): 0.2222
Epoch  13/100 | Loss: 0.031511 | Val F1 (95%): 0.2222
Early stopping. Best val F1: 0.2346


In [62]:
def find_best_threshold(scores, labels, percentiles=(95, 99.99), num=200):
    low, high = np.percentile(scores, percentiles)
    best_f1 = 0
    best_th = low
    for th in np.linspace(low, high, num):
        pred = (scores > th).astype(int)
        f1 = f1_score(labels, pred)
        if f1 > best_f1:
            best_f1 = f1
            best_th = th
    return best_th, best_f1

val_scores = predict_anomaly_scores(model, val_windows, val_contexts)
threshold, val_best_f1 = find_best_threshold(val_scores, val_labels)
print(f"Optimal threshold = {threshold:.6f}")
print(f"Best validation F1 = {val_best_f1:.4f}")

# Apply to final test set
final_test_scores = predict_anomaly_scores(model, final_test_windows, final_test_contexts)
final_test_pred = (final_test_scores > threshold).astype(int)

Optimal threshold = 0.069158
Best validation F1 = 0.2345


In [63]:
print("\n" + "="*60)
print("CLASSIFICATION REPORT (Final Test Set)")
print("="*60)
print(classification_report(final_test_labels, final_test_pred, target_names=['normal', 'anomaly']))

cm = confusion_matrix(final_test_labels, final_test_pred)
print("\nConfusion Matrix:\n", cm)

auc = roc_auc_score(final_test_labels, final_test_scores)
print(f"\nROC-AUC: {auc:.4f}")

prec, rec, f1, _ = precision_recall_fscore_support(final_test_labels, final_test_pred, average=None)
print("\nPer‑class metrics:")
print(f"Normal  - Precision: {prec[0]:.4f}, Recall: {rec[0]:.4f}, F1: {f1[0]:.4f}")
print(f"Anomaly - Precision: {prec[1]:.4f}, Recall: {rec[1]:.4f}, F1: {f1[1]:.4f}")

overall_acc = (final_test_labels == final_test_pred).mean()
print(f"\nOverall accuracy: {overall_acc:.4f}")


CLASSIFICATION REPORT (Final Test Set)
              precision    recall  f1-score   support

      normal       0.95      0.97      0.96      1413
     anomaly       0.20      0.14      0.16        87

    accuracy                           0.92      1500
   macro avg       0.58      0.55      0.56      1500
weighted avg       0.90      0.92      0.91      1500


Confusion Matrix:
 [[1366   47]
 [  75   12]]

ROC-AUC: 0.6368

Per‑class metrics:
Normal  - Precision: 0.9480, Recall: 0.9667, F1: 0.9573
Anomaly - Precision: 0.2034, Recall: 0.1379, F1: 0.1644

Overall accuracy: 0.9187
